In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col , sum , count
spark = SparkSession.builder.appName('EcommerceOrderPipeline').getOrCreate()

In [4]:
orders = spark.read.csv('orders.csv',header=True,inferSchema=True)
orders.show()

+--------+-----------+--------+--------+----------+---------+
|order_id|customer_id| product|quantity|unit_price|   status|
+--------+-----------+--------+--------+----------+---------+
|   O1001|       C101|  Laptop|       1|     55000|Delivered|
|   O1002|       C102|   Mouse|       3|       500|Delivered|
|   O1003|       C103| Monitor|       2|     12000|  Shipped|
|   O1004|       C101|Keyboard|       2|      1200|Delivered|
|   O1005|       C104|  Laptop|       1|     55000|Cancelled|
|   O1006|       C105|   Mouse|       5|       500|Delivered|
|   O1007|       C102| Monitor|       1|     12000|Delivered|
|   O1008|       C106|Keyboard|       3|      1200|  Shipped|
|   O1009|       C103|  Laptop|       2|     55000|Delivered|
|   O1010|       C105|   Mouse|       2|       500|Delivered|
+--------+-----------+--------+--------+----------+---------+



In [5]:
customers = spark.read.csv('customers.csv',header=True,inferSchema=True)
customers.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|       C101|         Arun|Bengaluru|
|       C102|        Priya|  Chennai|
|       C103|         Ravi|Hyderabad|
|       C104|        Meena|   Mumbai|
|       C105|        Kiran|     Pune|
|       C106|        Divya|  Chennai|
+-----------+-------------+---------+



In [7]:
orders.printSchema()
customers.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- status: string (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)



In [9]:
orders.groupBy('order_id').count().filter(col('count')>1).show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [13]:
orders.filter(col('customer_id').isNull()).show()

+--------+-----------+-------+--------+----------+------+
|order_id|customer_id|product|quantity|unit_price|status|
+--------+-----------+-------+--------+----------+------+
+--------+-----------+-------+--------+----------+------+



In [16]:
orders = orders.withColumn('order_amount',col('quantity')*col('unit_price'))

In [18]:
orders_with_customers = orders.join(customers,on='customer_id',how='left')

In [19]:
orders_with_customers.show()

+-----------+--------+--------+--------+----------+---------+------------+-------------+---------+
|customer_id|order_id| product|quantity|unit_price|   status|order_amount|customer_name|     city|
+-----------+--------+--------+--------+----------+---------+------------+-------------+---------+
|       C101|   O1001|  Laptop|       1|     55000|Delivered|       55000|         Arun|Bengaluru|
|       C102|   O1002|   Mouse|       3|       500|Delivered|        1500|        Priya|  Chennai|
|       C103|   O1003| Monitor|       2|     12000|  Shipped|       24000|         Ravi|Hyderabad|
|       C101|   O1004|Keyboard|       2|      1200|Delivered|        2400|         Arun|Bengaluru|
|       C104|   O1005|  Laptop|       1|     55000|Cancelled|       55000|        Meena|   Mumbai|
|       C105|   O1006|   Mouse|       5|       500|Delivered|        2500|        Kiran|     Pune|
|       C102|   O1007| Monitor|       1|     12000|Delivered|       12000|        Priya|  Chennai|
|       C1

In [56]:
revenue_by_city_product = orders_with_customers.groupBy('city','product').agg(sum('order_amount'))
revenue_by_city_product.show()

+---------+--------+-----------------+
|     city| product|sum(order_amount)|
+---------+--------+-----------------+
|  Chennai|Keyboard|             3600|
|Hyderabad| Monitor|            24000|
|  Chennai|   Mouse|             1500|
|Bengaluru|  Laptop|            55000|
|  Chennai| Monitor|            12000|
|   Mumbai|  Laptop|            55000|
|Bengaluru|Keyboard|             2400|
|     Pune|   Mouse|             3500|
|Hyderabad|  Laptop|           110000|
+---------+--------+-----------------+



In [57]:
orders_with_customers.groupBy('status').agg(sum('order_amount')).show

<bound method DataFrame.show of DataFrame[status: string, sum(order_amount): bigint]>

In [41]:
delivered_orders = orders_with_customers.filter(col('status') == 'Delivered')
delivered_orders.show()

+-----------+--------+--------+--------+----------+---------+------------+-------------+---------+
|customer_id|order_id| product|quantity|unit_price|   status|order_amount|customer_name|     city|
+-----------+--------+--------+--------+----------+---------+------------+-------------+---------+
|       C101|   O1001|  Laptop|       1|     55000|Delivered|       55000|         Arun|Bengaluru|
|       C102|   O1002|   Mouse|       3|       500|Delivered|        1500|        Priya|  Chennai|
|       C101|   O1004|Keyboard|       2|      1200|Delivered|        2400|         Arun|Bengaluru|
|       C105|   O1006|   Mouse|       5|       500|Delivered|        2500|        Kiran|     Pune|
|       C102|   O1007| Monitor|       1|     12000|Delivered|       12000|        Priya|  Chennai|
|       C103|   O1009|  Laptop|       2|     55000|Delivered|      110000|         Ravi|Hyderabad|
|       C105|   O1010|   Mouse|       2|       500|Delivered|        1000|        Kiran|     Pune|
+---------

In [49]:
top_customers = delivered_orders.orderBy('order_amount',ascending=False).limit(3)
top_customers.show()

+-----------+--------+-------+--------+----------+---------+------------+-------------+---------+
|customer_id|order_id|product|quantity|unit_price|   status|order_amount|customer_name|     city|
+-----------+--------+-------+--------+----------+---------+------------+-------------+---------+
|       C103|   O1009| Laptop|       2|     55000|Delivered|      110000|         Ravi|Hyderabad|
|       C101|   O1001| Laptop|       1|     55000|Delivered|       55000|         Arun|Bengaluru|
|       C102|   O1007|Monitor|       1|     12000|Delivered|       12000|        Priya|  Chennai|
+-----------+--------+-------+--------+----------+---------+------------+-------------+---------+



In [50]:
orders_with_customers.write.mode('overwrite').parquet('output3/cleaned_orders')

In [51]:
revenue_by_city_product.write.mode('overwrite').parquet('output3/revenue_by_city_product')
delivered_orders.write.mode('overwrite').parquet('output3/delivered_orders')
top_customers.write.mode('overwrite').parquet('output3/top_customers')

In [52]:
spark.read.parquet('output3/revenue_by_city_product').show()

+---------+--------+-----------------+
|     city| product|sum(order_amount)|
+---------+--------+-----------------+
|  Chennai|Keyboard|             3600|
|Hyderabad| Monitor|            24000|
|  Chennai|   Mouse|             1500|
|Bengaluru|  Laptop|            55000|
|  Chennai| Monitor|            12000|
|   Mumbai|  Laptop|            55000|
|Bengaluru|Keyboard|             2400|
|     Pune|   Mouse|             3500|
|Hyderabad|  Laptop|           110000|
+---------+--------+-----------------+

